In [28]:
# Cell 1 — Setup and load all metrics
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.train_linear import train_linear_model
from src.models.train_random_forest import train_random_forest
from src.models.train_xgboost import train_xgboost

In [29]:
# Cell 2 — Train all three models (or load saved metrics if already run)
linear_model, linear_metrics = train_linear_model()
rf_model, rf_metrics = train_random_forest()
xgb_model, xgb_metrics = train_xgboost()

Train: 1,231 rows (2021–2023)
Val:   451 rows (2024)
Test:  681 rows (2025 onward)
Linear Regression (Ridge): MAE=0.394s | RMSE=0.556s | MedianAE=0.269s
Saved model to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\artefacts\linear_ridge.pkl
Saved metrics to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\metrics\linear_ridge_metrics.csv
Train: 1,231 rows (2021–2023)
Val:   451 rows (2024)
Test:  681 rows (2025 onward)
Random Forest: MAE=0.405s | RMSE=0.536s | MedianAE=0.312s
Saved model to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\artefacts\random_forest.pkl
Saved metrics to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\metrics\random_forest_metrics.csv
Train: 1,231 rows (2021–2023)
Val:   451 rows (2024)
Test:  681 rows (2025 onward)
XGBoost (default params): MAE=0.439s | RMSE=0.555s | MedianAE=0.368s
Saved model to C:\Users\siraj\OneDrive\Project Portfolio\f1-

In [30]:
# Cell 3 — Load Baseline A's metric from Phase 5 for comparison
baseline_a_mae = pd.read_csv(PROJECT_ROOT / "models/metrics/baseline_best.csv")["MAE"].iloc[0]

comparison = pd.DataFrame([
    {"Model": "Baseline A (Driver Rolling)", "MAE": baseline_a_mae},
    {"Model": linear_metrics["Model"], "MAE": linear_metrics["MAE"]},
    {"Model": rf_metrics["Model"], "MAE": rf_metrics["MAE"]},
    {"Model": xgb_metrics["Model"], "MAE": xgb_metrics["MAE"]},
])

comparison

,Model,MAE
0,Baseline A (Driver Rolling),0.8353
1,Linear Regression (Ridge),0.3943
2,Random Forest,0.4053
3,XGBoost (default params),0.4389


In [31]:
# Cell 4 — Visual comparison against the benchmark
fig = px.bar(
    comparison, x="Model", y="MAE",
    title="Model Comparison — MAE vs Baseline A Benchmark",
    labels={"MAE": "MAE (seconds)"},
    text="MAE",
    color="Model"
)
fig.add_hline(
    y=baseline_a_mae, line_dash="dash", line_color="red",
    annotation_text="Baseline A", annotation_position="top left"
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [32]:
# Cell 5 — Residual analysis for the best model (assume XGBoost wins)
from src.models.data_split import load_features, time_based_split
from src.models.train_utils import get_feature_columns, prepare_xy

df = load_features()
train, val, test = time_based_split(df)
feature_cols = get_feature_columns(df)
X_val, y_val = prepare_xy(val, feature_cols)

val = val.copy()
val["Prediction"] = xgb_model.predict(X_val)
val["Residual"] = val["DeltaToFastest_s"] - val["Prediction"]

fig = px.scatter(
    val, x="Prediction", y="Residual",
    title="XGBoost Residuals vs Predictions",
    labels={"Prediction": "Predicted Delta (s)", "Residual": "Residual (Actual - Predicted)"},
    opacity=0.5
)
fig.add_hline(y=0, line_dash="dash", line_color="white")
fig.show()

Train: 1,231 rows (2021–2023)
Val:   451 rows (2024)
Test:  681 rows (2025 onward)


In [33]:
# Cell 6 — Error breakdown by circuit for the winning model
from src.models.evaluate import evaluate_by_group

by_circuit = evaluate_by_group(val, "DeltaToFastest_s", "Prediction", "EventName")
by_circuit.head(10)

,EventName,MAE
11,Hungarian Grand Prix,0.679255
3,Azerbaijan Grand Prix,0.662065
5,Belgian Grand Prix,0.648683
14,Las Vegas Grand Prix,0.526984
9,Dutch Grand Prix,0.495636
17,Monaco Grand Prix,0.485238
8,Chinese Grand Prix,0.456659
18,Qatar Grand Prix,0.452053
21,Spanish Grand Prix,0.428865
20,Singapore Grand Prix,0.423533


In [34]:
# Cell 7 — Error breakdown by team
by_team = evaluate_by_group(val, "DeltaToFastest_s", "Prediction", "Team")
by_team

,Team,MAE
0,Alpine,0.582251
5,McLaren,0.569565
4,Kick Sauber,0.481414
3,Haas F1 Team,0.449249
7,RB,0.435516
9,Williams,0.430143
6,Mercedes,0.421823
8,Red Bull Racing,0.393166
1,Aston Martin,0.368056
2,Ferrari,0.264223


In [35]:
import pandas as pd

importances = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": xgb_model.named_steps["model"].feature_importances_
}).sort_values("Importance", ascending=False)

importances.head(15)

,Feature,Importance
2,DriverRollingDelta_10,0.154617
18,CompoundOrdinal,0.147498
5,TeamRollingDelta_5,0.063502
1,DriverRollingDelta_5,0.063203
16,Rainfall,0.050150
14,Humidity,0.049883
6,TeamCareerMedianDelta,0.049539
11,Altitude_m,0.048235
12,TrackTemp,0.042104
15,WindSpeed,0.041695


In [36]:
from src.models.data_split import load_features, time_based_split
from src.models.train_utils import get_feature_columns, prepare_xy
from src.models.evaluate import evaluate_predictions

df = load_features()
train, val, test = time_based_split(df)

feature_cols = get_feature_columns(df)
X_train, y_train = prepare_xy(train, feature_cols)
X_val, y_val = prepare_xy(val, feature_cols)

train_preds = xgb_model.predict(X_train)
val_preds = xgb_model.predict(X_val)

evaluate_predictions(y_train, train_preds, label="XGBoost — TRAIN")
evaluate_predictions(y_val, val_preds, label="XGBoost — VAL")

Train: 1,231 rows (2021–2023)
Val:   451 rows (2024)
Test:  681 rows (2025 onward)
XGBoost — TRAIN: MAE=0.162s | RMSE=0.220s | MedianAE=0.122s
XGBoost — VAL: MAE=0.439s | RMSE=0.555s | MedianAE=0.368s


{'Model': 'XGBoost — VAL', 'MAE': 0.4389, 'RMSE': 0.5547, 'MedianAE': 0.3675}

In [37]:
val = val.copy()
val["Prediction"] = xgb_model.predict(X_val)
val["AbsError"] = (val["DeltaToFastest_s"] - val["Prediction"]).abs()

worst_predictions = val.sort_values("AbsError", ascending=False).head(15)
worst_predictions[["Year", "RoundNumber", "EventName", "Driver", "Team",
                    "DeltaToFastest_s", "Prediction", "AbsError"]]

,Year,RoundNumber,EventName,Driver,Team,DeltaToFastest_s,Prediction,AbsError
1476,2024,13,Hungarian Grand Prix,RUS,Mercedes,2.741,0.637791,2.103209
1552,2024,17,Azerbaijan Grand Prix,OCO,Alpine,3.139,1.395815,1.743185
1466,2024,13,Hungarian Grand Prix,GAS,Alpine,2.939,1.244695,1.694305
1521,2024,15,Dutch Grand Prix,ZHO,Kick Sauber,3.588,1.934786,1.653214
1472,2024,13,Hungarian Grand Prix,OCO,Alpine,2.822,1.271034,1.550966
1551,2024,17,Azerbaijan Grand Prix,NOR,McLaren,2.244,0.693148,1.550852
1500,2024,14,Belgian Grand Prix,TSU,RB,3.434,1.915471,1.518529
1473,2024,13,Hungarian Grand Prix,PER,Red Bull Racing,2.659,1.198606,1.460394
1488,2024,14,Belgian Grand Prix,HUL,Haas F1 Team,3.149,1.714266,1.434734
1614,2024,20,Mexico City Grand Prix,PER,Red Bull Racing,1.665,0.317900,1.347100


In [38]:
val[val["DeltaToFastest_s"] > 5][["Year", "RoundNumber", "EventName", "Driver", "Team", "DeltaToFastest_s"]]

,Year,RoundNumber,EventName,Driver,Team,DeltaToFastest_s


In [39]:
print(f"XGBoost — TEST: MAE={test_mae:.3f}s | RMSE={test_rmse:.3f}s | MedianAE={test_medianae:.3f}s")

NameError: name 'test_mae' is not defined